# exp02 U-Net residual sobre la grilla WRF

Entradas en la grilla 3 km (70x35 con padding a 72x36): GFS t-2, t-1, t interpolado por coordenadas (bilineal, como el starter) y normalizado, DEM, DEM menos DEM suavizado 9x9, pendiente y hora seno/coseno.
Objetivo: residuo WRF - GFS(t) interpolado, escalado por la desviación del residuo por variable.
Pérdida: promedio_v(MSE_v / REF_v^2) en unidades físicas. Validación temporal: times >= 2024061600.
El entrenamiento corre por épocas hasta agotar un presupuesto de tiempo (variable TIME_BUDGET_MIN).

In [1]:
import glob, os, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
torch.set_num_threads(2)
torch.manual_seed(0); np.random.seed(0)

def find(name):
    hits = glob.glob(f'../data/**/{name}', recursive=True) + glob.glob(f'data/**/{name}', recursive=True) + glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, name
    return hits[0]

tr, te, st = np.load(find('train.npz')), np.load(find('test.npz')), np.load(find('static.npz'))
Xtr, Ytr, ttr = tr['X'].astype(np.float32), tr['Y'].astype(np.float32), tr['times'].astype(str)
Xte, tte = te['X'].astype(np.float32), te['times'].astype(str)
dem = st['dem'].astype(np.float32)
VARS = ['RH', 'T', 'U', 'V']
REF = np.array([25.360, 2.0652, 2.6776, 2.0876], np.float32)
TIME_BUDGET_MIN = float(os.environ.get('TIME_BUDGET_MIN', '11'))
MAX_EPOCHS = int(os.environ.get('MAX_EPOCHS', '40'))
WIDTH = int(os.environ.get('WIDTH', '32'))
DEADLINE = os.environ.get('DEADLINE')  # 'HH:MM' hora local: corta el entrenamiento al llegar
def past_deadline():
    return DEADLINE is not None and time.strftime('%H:%M') >= DEADLINE
hour = lambda t: np.array([int(k[-2:]) for k in t])
val = ttr >= '2024061600'; trn = ~val
print(Xtr.shape, Ytr.shape, Xte.shape, 'train', trn.sum(), 'val', val.sum())

(2935, 17, 13, 4) (2935, 70, 35, 4) (792, 17, 13, 4) train 2599 val 336


In [2]:
def windows(X, t):
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        for j, lag in enumerate([2, 1, 0]):
            idx = pos.get(k - pd.Timedelta(hours=lag), None)
            out[i, j] = X[idx] if idx is not None else X[i]
    return out

Wtr, Wte = windows(Xtr, ttr), windows(Xte, tte)

# Coordenadas WRF dentro de la grilla GFS (GFS fila 0 = norte, WRF fila 0 = sur)
glat, glon, wlat, wlon = st['gfs_lats'], st['gfs_lons'], st['wrf_lats'], st['wrf_lons']
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32)

def interp(W, bs=256):
    # W (N,3,17,13,4) -> (N,12,70,35) en unidades físicas, canal = lag*4 + var
    out = np.empty((len(W), 12, 70, 35), np.float32)
    with torch.no_grad():
        for s in range(0, len(W), bs):
            x = torch.tensor(W[s:s+bs]).permute(0, 1, 4, 2, 3).reshape(-1, 12, 17, 13)
            out[s:s+bs] = F.grid_sample(x, GRID.expand(len(x), -1, -1, -1), mode='bilinear', align_corners=True).numpy()
    return out

Gtr, Gte = interp(Wtr), interp(Wte)
GtT = Gtr[:, 8:12].transpose(0, 2, 3, 1)   # GFS(t) interpolado, (N,70,35,4)
GteT = Gte[:, 8:12].transpose(0, 2, 3, 1)
print(Gtr.shape, Gte.shape)

(2935, 12, 70, 35) (792, 12, 70, 35)


In [3]:
# Estáticos: DEM, DEM - suavizado 9x9, pendiente
d = torch.tensor(dem)[None, None]
smooth = F.avg_pool2d(F.pad(d, (4, 4, 4, 4), mode='replicate'), 9, stride=1)[0, 0].numpy()
dlat_m = np.abs(np.diff(wlat).mean()) * 111e3
dlon_m = np.abs(np.diff(wlon).mean()) * 111e3 * np.cos(np.deg2rad(wlat.mean()))
gyd, gxd = np.gradient(dem, dlat_m, dlon_m)
slope = np.sqrt(gxd**2 + gyd**2)
stat = np.stack([dem, dem - smooth, slope]).astype(np.float32)
stat = (stat - stat.mean((1, 2), keepdims=True)) / stat.std((1, 2), keepdims=True)

# Normalización de entradas (por variable, estadísticas de train)
mu_g = Gtr[trn].reshape(-1, 3, 4, 70, 35).mean((0, 1, 3, 4)); sd_g = Gtr[trn].reshape(-1, 3, 4, 70, 35).std((0, 1, 3, 4))
mu_g12 = np.tile(mu_g, 3)[None, :, None, None]; sd_g12 = np.tile(sd_g, 3)[None, :, None, None]
# Residuo y su escala
R = Ytr - GtT
sd_r = R[trn].std((0, 1, 2)).astype(np.float32)
LW = (sd_r**2 / REF**2); LW = LW / LW.mean() * 1.0   # pesos de la pérdida en espacio escalado (proporcionales)
print('sd_g', sd_g.round(3), 'sd_r', sd_r.round(3), 'pesos', LW.round(3))

sd_g 

[15.511  5.507  1.705  1.152] sd_r [14.34   3.472  2.412  2.306] pesos [0.247 2.184 0.627 0.943]


In [4]:
PH, PW = 72, 36
def make_input(G, t):
    x = (G - mu_g12) / sd_g12
    ang = 2 * np.pi * hour(t) / 24
    hh = np.stack([np.sin(ang), np.cos(ang)], 1)[:, :, None, None] * np.ones((1, 1, 70, 35), np.float32)
    x = np.concatenate([x, np.broadcast_to(stat[None], (len(G), 3, 70, 35)), hh], 1).astype(np.float32)
    return x   # (N,17,70,35)

Itr, Ite = make_input(Gtr, ttr), make_input(Gte, tte)
Ttr = (R / sd_r).transpose(0, 3, 1, 2).astype(np.float32)   # (N,4,70,35)

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.GELU(),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.GELU())

class UNet(nn.Module):
    def __init__(self, cin=17, w=32):
        super().__init__()
        self.e1, self.e2, self.b = block(cin, w), block(w, 2*w), block(2*w, 4*w)
        self.u2, self.d2 = nn.ConvTranspose2d(4*w, 2*w, 2, 2), block(4*w, 2*w)
        self.u1, self.d1 = nn.ConvTranspose2d(2*w, w, 2, 2), block(2*w, w)
        self.out = nn.Conv2d(w, 4, 1)
    def forward(self, x):                       # x (B,17,70,35)
        x = F.pad(x, (0, PW - 35, 0, PH - 70), mode='replicate')
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); b = self.b(F.max_pool2d(e2, 2))
        d2 = self.d2(torch.cat([self.u2(b), e2], 1)); d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)[:, :, :70, :35]

model = UNet(w=WIDTH)
n_params = sum(p.numel() for p in model.parameters())
print('parámetros:', n_params)

parámetros: 119748


In [5]:
LWt = torch.tensor(LW)[None, :, None, None]
def predict(model, I, GT, bs=128):
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(I), bs):
            out.append(model(torch.tensor(I[s:s+bs])).permute(0, 2, 3, 1).numpy())
    return GT + np.concatenate(out) * sd_r

def rmse_v(P, Y): return np.sqrt(((P - Y)**2).mean((0, 1, 2)))
def score(P, Y): r = rmse_v(P, Y); return 1 - (r / REF).mean(), r

s0, r0 = score(GtT[val], Ytr[val])
print('Referencia GFS(t) interpolado en val: score', round(s0, 4), dict(zip(VARS, r0.round(3))))

itr = np.where(trn)[0]; BS = 32
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
best, best_state, best_ep, hist, sched = -9e9, None, 0, [], None
t_start = time.time()
for ep in range(MAX_EPOCHS):
    model.train(); np.random.shuffle(itr); t0 = time.time(); tl = 0
    for s in range(0, len(itr), BS):
        b = np.sort(itr[s:s+BS])
        x, y = torch.tensor(Itr[b]), torch.tensor(Ttr[b])
        loss = (((model(x) - y)**2) * LWt).mean()
        opt.zero_grad(); loss.backward(); opt.step(); tl += loss.item() * len(b)
        if sched: sched.step()
        if past_deadline(): print('corte por hora en el lote', s, flush=True); break
    dt = time.time() - t0
    sc, rv = score(predict(model, Itr[val], GtT[val]), Ytr[val])
    hist.append(dict(epoca=ep+1, loss=tl/len(itr), score_val=sc, **{f'RMSE_{v}': r for v, r in zip(VARS, rv)}, seg=dt))
    if sc > best: best, best_ep, best_state = sc, ep+1, {k: v.clone() for k, v in model.state_dict().items()}
    print(f'época {ep+1:02d} loss {tl/len(itr):.4f} score val {sc:.4f} {dict(zip(VARS, rv.round(3)))} {dt:.0f}s', flush=True)
    if ep == 0:   # fija el número de épocas con el tiempo medido y arma un coseno
        n_ep = int(min(MAX_EPOCHS, max(2, (TIME_BUDGET_MIN * 60) // (dt * 1.15))))
        steps = (n_ep - 1) * int(np.ceil(len(itr) / BS))
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, steps), eta_min=5e-5)
        print('épocas planificadas:', n_ep)
    if ep + 1 >= n_ep or past_deadline(): break
model.load_state_dict(best_state)
print('Mejor época', best_ep, 'score val', round(best, 4))

Referencia GFS(t) interpolado en val: score -0.1637 {'RH': np.float32(15.439), 'T': np.float32(3.759), 'U': np.float32(2.553), 'V': np.float32(2.656)}


corte por hora en el lote 0


época 01 loss 0.0143 score val -0.1791 {'RH': np.float32(15.746), 'T': np.float32(3.783), 'U': np.float32(2.574), 'V': np.float32(2.718)} 18s


épocas planificadas: 17
Mejor época 1 score val -0.1791


In [6]:
os.makedirs('results', exist_ok=True); os.makedirs('../models', exist_ok=True)
P = predict(model, Itr[val], GtT[val]); sc, rv = score(P, Ytr[val])
h = pd.DataFrame(hist); h.to_csv('results/exp02_historial.csv', index=False)
res = pd.DataFrame([dict(experimento='exp02_unet', score_val=round(sc, 4), score_gfs_interp=round(s0, 4),
                         **{f'RMSE_{v}': round(float(r), 4) for v, r in zip(VARS, rv)},
                         mejor_epoca=best_ep, epocas=len(hist), seg_por_epoca=round(h.seg.mean(), 1),
                         parametros=n_params, anchura=WIDTH, niveles=2)])
res.to_csv('results/exp02_resultados.csv', index=False)
torch.save(dict(state_dict=model.state_dict(), arch=dict(cin=17, w=WIDTH, PH=PH, PW=PW),
                mu_g=mu_g, sd_g=sd_g, sd_r=sd_r, stat=stat, grid=GRID.numpy(), REF=REF, VARS=VARS,
                notas='entrada: ventanas (N,3,17,13,4) -> interp bilineal con grid -> normaliza por variable con mu_g/sd_g (repetido x3) + stat + sin/cos hora UTC; salida = GFS(t) interp + pred*sd_r; fila 0 = sur'),
           '../models/exp02_unet.pt')
res

,experimento,score_val,score_gfs_interp,RMSE_RH,RMSE_T,RMSE_U,RMSE_V,mejor_epoca,epocas,seg_por_epoca,parametros,anchura,niveles
0,exp02_unet,-0.1791,-0.1637,15.7462,3.7828,2.5744,2.7184,1,1,17.6,119748,16,2


In [7]:
sample = pd.read_csv(find('sample_submission.csv'))
PT = predict(model, Ite, GteT)
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti = parts[0].map(pos).values; r = parts[1].astype(int).values; c = parts[2].astype(int).values
sub = pd.DataFrame({'id': sample['id']})
for k, v in enumerate(VARS):
    sub[v] = PT[ti, r, c, k].round(3)
sub.to_csv('results/exp02_submission.csv', index=False)
print(sub.shape, sub.isna().sum().sum()); sub.describe()

(646800, 5) 0


,RH,T,U,V
count,646800.000000,646800.000000,646800.000000,646800.000000
mean,61.802486,281.457520,-1.369339,-0.016749
std,19.954727,5.474281,1.835178,1.263580
min,10.770000,264.355011,-8.084000,-4.966000
25%,45.646000,277.485992,-2.669000,-0.810000
50%,63.227001,281.148010,-1.612000,-0.150000
75%,79.242996,285.553986,-0.456000,0.615000
max,100.171997,298.972992,5.234000,6.983000
